# 第 5 章 Notebook：子 Agent 委派与上下文隔离

对应课程章节：[第 5 章：子 Agent 与上下文隔离](../../content/ch05-subagents.md)。

本实验只使用本地固定的研究资料，不调用搜索服务。主 Agent 把比较任务交给声明式 `researcher`，子 Agent 读取资料、写入研究记录并回传结论。我们分别检查**主 Agent 的消息**和**共享的文件状态**，看清两个边界。

## 学习目标

运行结束后，你应该能够：

1. 用字典声明 `researcher`，并让主 Agent 通过 `task` 委派；
2. 从 `AIMessage.tool_calls` 与 `ToolMessage` 检查委派和结果回传；
3. 证明子 Agent 的内部工具记录不自动进入主 Agent 的 `messages`；
4. 证明默认 `StateBackend` 的文件状态会回到主 Agent，且主 Agent 可以主动读取它。

本章只讨论进程内的同步子 Agent；第 6 章另讲异步子 Agent。

## 运行环境与依赖

本 Notebook 按以下版本编写。代码格会打印你实际安装的版本；提交 PR 时还需记录实际运行环境。

| 项目 | 版本 |
|---|---|
| 操作系统 | macOS（Apple Silicon）；Linux / WSL 可按相同步骤运行 |
| Python | 3.12 |
| deepagents | 0.7.15 |
| langchain | 1.4.2 |
| langgraph | 1.2.11 |
| langchain-openai | 1.6.2 |

安装与内核选择见 [Notebook 索引](../README.md)。本 Notebook 从第一格独立运行，无需先执行第 1 章。

需要一个支持工具调用的 OpenAI 兼容模型 API。默认沿用课程的 SiliconFlow 配置：`SILICONFLOW_API_KEY`、可选的 `MODEL_NAME` 和 `SILICONFLOW_BASE_URL`。也可设置 `DEEPSEEK_API_KEY` 使用 DeepSeek；默认模型 `deepseek-flash`、接口 `https://api.deepseek.com`，可用 `MODEL_NAME` 覆盖。其他 OpenAI 兼容服务可同时设置 `MODEL_API_KEY`、`MODEL_BASE_URL` 和 `MODEL_NAME`。可在终端设置环境变量，或在仓库根目录创建未提交的 `.env`。例如使用课程默认服务时写入 `SILICONFLOW_API_KEY=你的密钥`；使用其他服务时同时设置 `MODEL_API_KEY`、`MODEL_BASE_URL` 和 `MODEL_NAME`。**不要把密钥写入 Notebook。** 除模型 API 外，不需要 Tavily、数据库或沙箱；模型请求需要网络，费用与额度以服务商实际情况为准。

## 0. 检查版本并初始化模型

下面只打印版本、模型名和密钥是否存在，不打印密钥本身。若模型不支持工具调用，后面的可检查断言会明确失败。

下方保留了使用 DeepSeek `deepseek-flash` 从第一格执行得到的输出；密钥没有写入 Notebook。模型的回答措辞可能变化，请以工具调用、状态和断言为准。

In [1]:
import importlib.metadata as metadata
import platform

for package in ("deepagents", "langchain", "langgraph", "langchain-openai"):
    print(f"{package}=={metadata.version(package)}")
print("操作系统:", platform.system())
print("Python:", platform.python_version())

deepagents==0.7.15
langchain==1.4.2
langgraph==1.2.11
langchain-openai==1.6.2
操作系统: Darwin
Python: 3.12.13


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

ROOT = next((path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (path / "content/ch05-subagents.md").exists()), None)
assert ROOT is not None, "请从仓库目录运行 Notebook"
load_dotenv(ROOT / ".env", override=False)
DEEPSEEK = bool(os.getenv("DEEPSEEK_API_KEY")) and not os.getenv("MODEL_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME", "deepseek-flash" if DEEPSEEK else "Qwen/Qwen2.5-7B-Instruct")
BASE_URL = os.getenv("MODEL_BASE_URL") or (
    "https://api.deepseek.com" if DEEPSEEK else os.getenv("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1")
)
API_KEY = os.getenv("MODEL_API_KEY") or os.getenv("DEEPSEEK_API_KEY") or os.getenv("SILICONFLOW_API_KEY")
if not API_KEY:
    raise RuntimeError("缺少 MODEL_API_KEY、DEEPSEEK_API_KEY 或 SILICONFLOW_API_KEY。请在仓库根目录的 .env 中配置后从头重跑。")

model = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL, temperature=0, timeout=60)
print("MODEL_NAME =", MODEL_NAME)
print("BASE_URL =", BASE_URL)
print("API_KEY = 已设置")

MODEL_NAME = deepseek-flash
BASE_URL = https://api.deepseek.com
API_KEY = 已设置


## 1. 定义可复现的本地资料工具

以下数字是**教学用的固定案例**，不代表真实产品测量。`lookup_case` 的入参和返回值会记录在 Python 列表中，供我们观察子 Agent 的内部工具调用；这份列表不会自动成为主 Agent 的消息。

In [3]:
from langchain_core.tools import tool

CASE_ID = "cache-plan"
REPORT_PATH = "/research/cache-choice.md"
CASE_MATERIAL = """[A] 实时汇总：同一组 1000 次请求中，p95 延迟 420 ms，错误率 0.4%。
[B] 每 5 分钟更新一次缓存：同一组请求中，p95 延迟 95 ms，错误率 0.6%。
[约束] p95 必须低于 150 ms；允许结果最多延迟 5 分钟；错误率必须低于 1%。"""
lookup_trace = []

@tool
def lookup_case(case_id: str) -> str:
    """读取本地固定的缓存方案研究材料。case_id 应为 cache-plan。"""
    result = CASE_MATERIAL if case_id == CASE_ID else f"未知案例：{case_id}"
    lookup_trace.append({"case_id": case_id, "result": result})
    return result

## 2. 声明 researcher 并创建主 Agent

`researcher` 有自己的描述、指令和 `lookup_case` 工具。主 Agent 不注册 `lookup_case`，只负责委派和接收结果。Deep Agents 仍会为双方装配文件工具；这里不引入 Skills、Todo 或异步任务，以便只观察本章的核心行为。

`task` 的实际参数名是 `description` 和 `subagent_type`。它由主 Agent 在运行时调用，不需要我们在 Python 中手动调用。

In [4]:
from deepagents import create_deep_agent

researcher = {
    "name": "researcher",
    "description": "研究本地 cache-plan 案例，比较 A、B 两种方案，并把证据写入文件。",
    "system_prompt": f"""你是研究员。处理 cache-plan 时：
1. 调用 lookup_case(case_id="{CASE_ID}") 获取资料；
2. 根据资料比较 A、B 与约束，使用 write_file 把短报告写入 {REPORT_PATH}；
3. 报告中保留 [A]、[B]、[约束] 三个资料标签，并给出结论；
4. 最终只返回一两句结论及文件路径，不复制整段原始资料。""",
    "tools": [lookup_case],
}

agent = create_deep_agent(
    model=model,
    tools=[],
    system_prompt=(
        "你是协调者。遇到 cache-plan 研究任务，必须调用 task，"
        "并指定 subagent_type=researcher。研究资料由 researcher 的 lookup_case 工具提供，"
        "报告写入虚拟文件；不要要求搜索磁盘。收到结果后简短转述，"
        "本轮不要调用 lookup_case 或 read_file。后续用户明确要求读文件时再调用 read_file。"
    ),
    subagents=[researcher],
)

## 3. 委派一次研究任务

预期主 Agent 先调用 `task`，随后收到子 Agent 的简短结论。模型的自然语言措辞可能变化；我们只依赖工具调用和状态来验证。

In [5]:
first_result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "请把 cache-plan 案例交给 researcher：比较 A、B 哪个满足约束，保存有证据的短报告，然后告诉我结论。",
    }]
})
print("本轮主 Agent 消息数:", len(first_result["messages"]))

本轮主 Agent 消息数: 4


### 3.1 检查 `task` 的输入与返回

主 Agent 的 `AIMessage.tool_calls` 记录它选择了谁、交代了什么任务；名称为 `task` 的 `ToolMessage` 是子 Agent 的回传。这里还打印本地工具日志中的输入和返回值，作为子 Agent 确实读取资料的证据。

In [6]:
from langchain_core.messages import AIMessage, ToolMessage
import textwrap

task_calls = [
    call
    for message in first_result["messages"]
    if isinstance(message, AIMessage)
    for call in message.tool_calls
    if call["name"] == "task"
]
task_results = [
    message
    for message in first_result["messages"]
    if isinstance(message, ToolMessage) and message.name == "task"
]
assert any(call["args"].get("subagent_type") == "researcher" for call in task_calls), (
    "主 Agent 没有委派给 researcher；请检查 MODEL_NAME 是否支持工具调用，以及主 Agent 提示词。"
)
assert task_results and str(task_results[-1].content).strip(), "task 没有返回可观察的结果。"
assert any(item["case_id"] == CASE_ID for item in lookup_trace), (
    "researcher 没有读取固定资料；请检查它是否调用 lookup_case。"
)

def show_text(label, value):
    print(f"\n{label}")
    for line in str(value).splitlines():
        if not line.strip():
            print()
            continue
        print(textwrap.fill(
            line, width=76, subsequent_indent="  ",
            break_long_words=True, break_on_hyphens=False,
        ))


print("委派对象:", task_calls[0]["args"]["subagent_type"])
show_text("task 任务说明", task_calls[0]["args"]["description"])
show_text("task 返回", task_results[-1].content)
for item in lookup_trace:
    show_text(f"lookup_case({item['case_id']}) 返回", item["result"])

委派对象: researcher

task 任务说明
研究本地 cache-plan 案例，比较方案 A 与方案 B 哪个满足约束，并用证据写一份简短报告。

请使用你的 lookup_case 工具获取案例资料（不要搜索磁盘，不要假设路径）。任务要求：
1. 读取该 cache-plan 案例的全部相关材料。
2. 明确案例中的约束条件（constraints），逐条列出。
3. 分别评估方案 A 和方案 B 是否满足每一条约束，给出证据（引用资料中的原文或数据点）。
4. 判定 A、B 哪个（或哪些）满足约束，并说明理由。
5. 把带有证据的简短报告写入文件（虚拟文件即可），报告应包含：结论、约束清单、A/B 逐条对照证据、以及依据来源。

最终请返回：报告文件的路径、一段简明的结论摘要（哪个方案满足约束、关键证据要点），以及任何不确定或存疑之处。

task 返回
报告已写入：`/research/cache-choice.md`

**结论摘要**：方案 B 满足全部约束，方案 A 不满足。关键证据：A 的 p95 延迟为 420 ms，远超 [约束] 要求的 150 ms
  上限（是上限的 2.8 倍），属硬性否决项；B 的 p95 为 95 ms（<150 ms）、错误率 0.6%（<1%），其每 5
  分钟刷新带来的最大 5 分钟陈旧度也落在“最多延迟 5 分钟”的允许范围内。A 虽错误率更低（0.4%）且实时，但延迟已直接违反约束。

**存疑**：①B 的 5 分钟陈旧度恰好等于约束上限边界，若“最多 5 分钟”按“严格小于”解释则需复核（如改为每 4
  分钟刷新）；②资料未给样本分组与 p95/错误率的置信区间，A vs B 的错误率差异（0.4% vs
  0.6%）是否显著不明；③成本、一致性等其他潜在约束在资料中未出现。

lookup_case(cache-plan) 返回
[A] 实时汇总：同一组 1000 次请求中，p95 延迟 420 ms，错误率 0.4%。
[B] 每 5 分钟更新一次缓存：同一组请求中，p95 延迟 95 ms，错误率 0.6%。
[约束] p95 必须低于 150 ms；允许结果最多延迟 5 分钟；错误率必须低于 1%。


### 3.2 检查消息上下文边界

子 Agent 的 `lookup_case` 和 `write_file` 调用已经发生，但它们的内部 `ToolMessage` 不会自动并入主 Agent 的 `messages`。主 Agent 看到的是自己的 `task` 调用及其结果。这里检查的是**消息轨迹**，并不意味着子 Agent 的文件无法共享。

In [7]:
parent_tools = [
    message.name
    for message in first_result["messages"]
    if isinstance(message, ToolMessage)
]
print("主 Agent 收到的工具结果名:", parent_tools)
assert "task" in parent_tools
assert "lookup_case" not in parent_tools
assert "write_file" not in parent_tools
print("已验证：子 Agent 的资料查询和写文件记录没有自动进入主 Agent 的消息。")

主 Agent 收到的工具结果名: ['task']
已验证：子 Agent 的资料查询和写文件记录没有自动进入主 Agent 的消息。


## 4. 检查共享的文件状态

默认 `StateBackend` 把文件存在本次 Agent 的状态中。子 Agent 写入文件后，文件更新会出现在主 Agent 的返回状态。文件内容是否正确可以直接检查，无须猜测模型的最终回答。此时主 Agent **尚未主动读取**文件内容。

In [8]:
report = first_result.get("files", {}).get(REPORT_PATH)
assert report is not None, f"researcher 未在共享状态中写入 {REPORT_PATH}。"
report_text = report["content"]
assert all(label in report_text for label in ("[A]", "[B]", "[约束]", "420", "95")), (
    "报告缺少固定资料标签或关键延迟数据；请检查 researcher 的 write_file 调用。"
)
from IPython.display import Markdown, display

print("共享文件:", REPORT_PATH)
display(Markdown(report_text))

共享文件: /research/cache-choice.md


# cache-plan 方案选择报告

## 结论

**方案 B 满足全部约束；方案 A 不满足（p95 延迟严重超标）。** 因此应选择 [B]。

## 约束清单（来自 [约束]）

1. C1 延迟：p95 必须低于 150 ms。
2. C2 新鲜度：允许结果最多延迟 5 分钟。
3. C3 正确性：错误率必须低于 1%。

## 逐条对照

| 约束 | [A] 实时汇总 | [B] 每 5 分钟更新一次缓存 |
| --- | --- | --- |
| C1 p95 < 150 ms | 420 ms → **不满足**（超出上限 270 ms，约为上限的 2.8 倍） | 95 ms → **满足**（低于上限 55 ms） |
| C2 最多延迟 5 分钟 | 实时汇总，延迟约 0 → 满足 | 每 5 分钟更新 → 最大延迟 5 分钟，恰好落在允许边界内 → 满足（边界情况，见存疑） |
| C3 错误率 < 1% | 0.4% → 满足 | 0.6% → 满足 |

## 证据要点

- [A] 实时汇总：同一组 1000 次请求中，p95 延迟 420 ms，错误率 0.4%。
  - 唯一优势是新鲜度（实时）；但延迟指标 420 ms 远远突破 150 ms 硬性上限，属不可修复的否决项。
- [B] 每 5 分钟更新一次缓存：同一组请求中，p95 延迟 95 ms，错误率 0.6%。
  - 以约 5 分钟的数据新鲜度换取延迟从 420 ms 降至 95 ms，仍在 150 ms 以内；错误率 0.6% < 1%，虽略高于 A，但仍有 0.4 个百分点的余量。

## 依据来源

- [A]：实时汇总方案数据（p95 420 ms，错误率 0.4%）。
- [B]：5 分钟定时缓存方案数据（p95 95 ms，错误率 0.6%）。
- [约束]：p95 < 150 ms；结果最多延迟 5 分钟；错误率 < 1%。

## 存疑之处

1. **C2 为边界值**：B 的“每 5 分钟更新一次”意味着最坏情况下数据恰有 5 分钟陈旧，正好等于“最多延迟 5 分钟”的上限。若约束的严格解释是“必须小于 5 分钟”，则 B 在此条上处于边界、需要复核（例如改为每 4 分钟刷新）。
2. **误差无置信区间**：两组数据被描述为“同一组请求”，但未给出样本分组、p95 的置信区间或错误率的统计波动范围，A 与 B 错误率（0.4% vs 0.6%）的差异可能不显著。
3. **未提供其他约束**：如成本、缓存一致性/失效策略、写入路径开销等均未在资料中出现，本报告只依据 [约束] 中列出的三条判定。


### 4.1 让主 Agent 主动读取文件

第二次调用把上一轮的 `messages` 和 `files` 显式传回图中，再请求 `read_file`。这不需要数据库或磁盘文件；它演示的是**同一研究流程中的状态交接**。读文件之后，该次 `read_file` 的返回才会进入主 Agent 的消息上下文。

上一格已展示完整报告。这里核对 `read_file` 读回的是同一份内容，只显示返回头和行数，避免重复打印整篇报告。

In [9]:
second_result = agent.invoke({
    "messages": [
        *first_result["messages"],
        {"role": "user", "content": f"请使用 read_file 读取 {REPORT_PATH}，再简短告诉我报告的结论。"},
    ],
    "files": first_result["files"],
})
new_messages = second_result["messages"][len(first_result["messages"]):]
read_results = [
    message
    for message in new_messages
    if isinstance(message, ToolMessage) and message.name == "read_file"
]
assert read_results, "主 Agent 未调用 read_file；请换用能可靠调用工具的 MODEL_NAME 后重跑。"
read_text = str(read_results[-1].content)
assert report_text.strip() in read_text, "read_file 读回的内容与共享状态中的报告不一致。"
print("read_file 返回头:", read_text.splitlines()[0])
print("已读回共享状态中的完整报告：", len(report_text.splitlines()), "行")
print("已验证：消息上下文隔离，文件状态共享；主 Agent 主动读取后才看到文件内容。")

read_file 返回头: @@ lines 1-38 of 38 @@
已读回共享状态中的完整报告： 38 行
已验证：消息上下文隔离，文件状态共享；主 Agent 主动读取后才看到文件内容。


## 预期现象、常见问题与清理

- **委派与回传**：第一轮应出现指向 `researcher` 的 `task` 调用和非空的 `task` 工具结果；`lookup_trace` 能显示本地资料工具的输入与返回。
- **消息隔离**：第一轮主 Agent 的工具结果中有 `task`，没有子 Agent 内部的 `lookup_case`、`write_file`。
- **文件共享**：第一轮返回状态包含 `/research/cache-choice.md`，第二轮主 Agent 可用 `read_file` 读取它。共享文件不等于自动共享消息上下文。
- **模型未委派或未读取**：确认 `MODEL_NAME` 支持工具调用，使用已验证的模型后重跑。断言故意失败，以免把未发生的委派当作成功实验。
- **密钥、网络与额度**：缺少密钥会在初始化时报错；`401` 检查密钥，连接或超时错误检查网络与接口地址；切换服务商时一并设置 `MODEL_API_KEY`、`MODEL_BASE_URL`、`MODEL_NAME`；实际费用依模型和平台而定。
- **清理**：默认 `StateBackend` 的文件保存在内存状态中，不会写到本机磁盘。重启内核并从头执行即可得到干净状态；本地 `.env` 不要提交。提交 Notebook 前删除或截断过长输出，并检查没有密钥、Token 或个人路径。

可在仓库根目录从头验证：

```bash
jupyter nbconvert --to notebook --execute notebooks/ch05/01-subagent-delegation.ipynb --output-dir /tmp --ExecutePreprocessor.timeout=600
```